# 📊 Sales & Demand Forecasting Demo

This notebook demonstrates the complete workflow for sales forecasting using the project modules.

## Workflow:
1. Data Loading & Preprocessing
2. Exploratory Data Analysis (EDA)
3. Model Training & Evaluation
4. Time-Series Forecasting
5. Results Export

## 1. Setup and Imports

In [ ]:
# Add parent directory to path
import sys
sys.path.append('..')

# Import modules
from src.data_preprocessing import DataPreprocessor
from src.eda import EDA
from src.model_training import ModelTrainer
from src.forecasting import Forecaster

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
%matplotlib inline

print("✓ All modules imported successfully!")

## 2. Create Sample Dataset

Let's create a synthetic sales dataset for demonstration purposes.

In [ ]:
# Create synthetic sales data
np.random.seed(42)

# Generate date range (2 years of daily data)
dates = pd.date_range(start='2022-01-01', end='2023-12-31', freq='D')
n_days = len(dates)

# Products and categories
products = ['Product A', 'Product B', 'Product C', 'Product D', 'Product E']
categories = ['Electronics', 'Clothing', 'Food', 'Books', 'Toys']
regions = ['North', 'South', 'East', 'West']
stores = ['Store1', 'Store2', 'Store3', 'Store4', 'Store5']

# Generate data with seasonality and trend
data = []
for i, date in enumerate(dates):
    # Add trend
    trend = i * 0.1
    
    # Add seasonality (weekly and yearly)
    weekly_seasonality = 50 * np.sin(2 * np.pi * i / 7)
    yearly_seasonality = 100 * np.sin(2 * np.pi * i / 365)
    
    # Base sales with noise
    base_sales = 200 + trend + weekly_seasonality + yearly_seasonality + np.random.normal(0, 30)
    
    # Generate multiple records per day
    for _ in range(np.random.randint(3, 8)):
        product = np.random.choice(products)
        category = categories[products.index(product)]
        units_sold = max(int(base_sales + np.random.normal(0, 50)), 10)
        price_per_unit = np.random.uniform(20, 100)
        revenue = units_sold * price_per_unit
        
        data.append({
            'Date': date,
            'Product_Name': product,
            'Category': category,
            'Units_Sold': units_sold,
            'Revenue': round(revenue, 2),
            'Region': np.random.choice(regions),
            'Store': np.random.choice(stores)
        })

df_raw = pd.DataFrame(data)

# Save to dataset folder
df_raw.to_csv('../dataset/sample_sales_data.csv', index=False)

print(f"✓ Created sample dataset with {len(df_raw)} records")
print(f"Date range: {df_raw['Date'].min()} to {df_raw['Date'].max()}")
print(f"\nFirst few rows:")
df_raw.head()

## 3. Data Preprocessing

In [ ]:
# Initialize preprocessor
preprocessor = DataPreprocessor('../dataset/sample_sales_data.csv')

# Run preprocessing pipeline
df_clean = preprocessor.preprocess(
    date_column='Date',
    create_lags=False  # Set to True if you want lag features
)

print("\n✓ Preprocessing completed!")
print(f"Final shape: {df_clean.shape}")

## 4. Exploratory Data Analysis

In [ ]:
# Initialize EDA
eda = EDA(df_clean, date_column='Date')

# Generate all visualizations
insights = eda.generate_all_visualizations(
    output_dir='../output',
    value_column='Units_Sold',
    revenue_column='Revenue'
)

## 5. Model Training

In [ ]:
# Initialize model trainer
trainer = ModelTrainer(df_clean, target_column='Units_Sold', date_column='Date')

# Train all models
results = trainer.train_all_models()

# Compare models
comparison_df = trainer.compare_models(save_path='../output/model_comparison.png')

print("\n✓ Model training completed!")

## 6. Visualize Model Performance

In [ ]:
# Plot predictions for best model
best_model = comparison_df.loc[comparison_df['R² Score'].idxmax(), 'Model']
print(f"Best Model: {best_model}")

trainer.plot_predictions(
    model_name=best_model,
    save_path=f'../output/{best_model.replace(" ", "_")}_predictions.png'
)

# Feature importance (for tree-based models)
if best_model in ['Random Forest', 'XGBoost']:
    importance_df = trainer.get_feature_importance(
        model_name=best_model,
        top_n=15,
        save_path=f'../output/{best_model.replace(" ", "_")}_feature_importance.png'
    )

## 7. Time-Series Forecasting with Prophet

In [ ]:
# Aggregate data by date for time-series forecasting
df_ts = df_clean.groupby('Date')['Units_Sold'].sum().reset_index()

# Initialize forecaster
forecaster = Forecaster(df_ts, target_column='Units_Sold', date_column='Date')

# Generate 30-day forecast using Prophet
forecast_prophet = forecaster.forecast_next_n_days(
    n_days=30,
    model='prophet',
    save_path='../output/prophet_forecast.png'
)

print("\n✓ Prophet forecast completed!")
print("\nForecast Summary:")
print(forecast_prophet.head(10))

## 8. Time-Series Forecasting with ARIMA

In [ ]:
# Generate 30-day forecast using ARIMA
forecaster_arima = Forecaster(df_ts, target_column='Units_Sold', date_column='Date')

forecast_arima = forecaster_arima.forecast_next_n_days(
    n_days=30,
    model='arima',
    save_path='../output/arima_forecast.png'
)

print("\n✓ ARIMA forecast completed!")
if forecast_arima is not None:
    print("\nForecast Summary:")
    print(forecast_arima.head(10))

## 9. Compare Forecasting Models

In [ ]:
# Compare both forecasts
if forecast_arima is not None and forecast_prophet is not None:
    forecaster.compare_forecasts(
        arima_forecast=forecast_arima,
        prophet_forecast=forecast_prophet,
        save_path='../output/forecast_comparison.png'
    )
    
    # Statistical comparison
    print("\nForecast Statistics:")
    print("\nProphet:")
    print(f"  Mean: {forecast_prophet['Forecast'].mean():.2f}")
    print(f"  Std:  {forecast_prophet['Forecast'].std():.2f}")
    print(f"  Min:  {forecast_prophet['Forecast'].min():.2f}")
    print(f"  Max:  {forecast_prophet['Forecast'].max():.2f}")
    
    print("\nARIMA:")
    print(f"  Mean: {forecast_arima['Forecast'].mean():.2f}")
    print(f"  Std:  {forecast_arima['Forecast'].std():.2f}")
    print(f"  Min:  {forecast_arima['Forecast'].min():.2f}")
    print(f"  Max:  {forecast_arima['Forecast'].max():.2f}")

## 10. Export Results

In [ ]:
# Save processed data
preprocessor.save_processed_data('../output/processed_data.csv')

# Save forecasts
if forecast_prophet is not None:
    forecaster.save_forecast(forecast_prophet, '../output/prophet_forecast.csv')

if forecast_arima is not None:
    forecaster_arima.save_forecast(forecast_arima, '../output/arima_forecast.csv')

# Save best model
trainer.save_model(best_model, f'../models/{best_model.replace(" ", "_")}_model.pkl')

print("\n✓ All results exported successfully!")
print("\nCheck the following directories:")
print("  - output/  : Visualizations and forecasts")
print("  - models/  : Trained ML models")

## 11. Business Insights Summary

In [ ]:
print("=" * 80)
print("BUSINESS INSIGHTS SUMMARY")
print("=" * 80)

# Data insights
print("\n📊 DATA OVERVIEW:")
print(f"  Total Records: {len(df_clean):,}")
print(f"  Date Range: {df_clean['Date'].min().strftime('%Y-%m-%d')} to {df_clean['Date'].max().strftime('%Y-%m-%d')}")
print(f"  Total Revenue: ${df_clean['Revenue'].sum():,.2f}")
print(f"  Total Units Sold: {df_clean['Units_Sold'].sum():,.0f}")

# Model insights
print("\n🤖 MODEL PERFORMANCE:")
print(f"  Best Model: {best_model}")
print(f"  Test R² Score: {results[best_model]['test_r2']:.4f}")
print(f"  Test MAE: {results[best_model]['test_mae']:.2f}")

# Forecast insights
print("\n🔮 FORECAST (Next 30 Days):")
if forecast_prophet is not None:
    print(f"  Prophet Average: {forecast_prophet['Forecast'].mean():.2f} units/day")
    print(f"  Expected Total: {forecast_prophet['Forecast'].sum():.0f} units")
    print(f"  Trend: {'Upward' if forecast_prophet['Forecast'].iloc[-1] > forecast_prophet['Forecast'].iloc[0] else 'Downward'}")

# EDA insights
print("\n📈 KEY PATTERNS:")
for insight in insights:
    print(f"  {insight}")

print("\n" + "=" * 80)

## 12. Next Steps

### Recommendations:

1. **Model Improvement:**
   - Try hyperparameter tuning for better performance
   - Add more lag features for time-series models
   - Experiment with ensemble methods

2. **Feature Engineering:**
   - Add external features (holidays, promotions, weather)
   - Create product-specific features
   - Incorporate market trends

3. **Advanced Forecasting:**
   - Try LSTM or GRU neural networks
   - Implement multi-variate forecasting
   - Add confidence intervals visualization

4. **Deployment:**
   - Build REST API for model serving
   - Create automated reporting pipeline
   - Set up monitoring and retraining schedule

---

**🎉 Congratulations!** You've completed the sales forecasting workflow!
